In [1]:
import os
import re
import fnmatch
import glob
import sys
module_path = os.path.abspath(os.path.join('./src/python'))
sys.path.append(module_path)
import afim
import xesmf             as xe
import numpy             as np
import pandas            as pd
import xarray            as xr
import cosima_cookbook   as cc
import matplotlib.pyplot as plt
import cartopy.crs       as ccrs
import cmocean
from dask.distributed import Client
from datetime         import datetime,timedelta
from dask.diagnostics       import ProgressBar
import importlib
%matplotlib inline

In [ ]:
importlib.reload(afim)

In [ ]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src','python', 'afim_on_gadi.json'))

In [ ]:
JRA = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/8XDAILY/JRA55_03hr_forcing_tx1_2005.nc')
JRA.spchmd.isel(time=0).plot(figsize=(15,10))

JRA55do

In [ ]:
huss = xr.open_dataset('/home/581/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_huss_2005_bilin.nc')
huss.isel(time=0).huss.plot(figsize=(15,10))

In [ ]:
prra = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_prra_2005.nc')
prsn = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_prsn_2005.nc')
rlds = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_rlds_2005.nc')
rsds = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_rsds_2005.nc')
tas = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_tas_2005.nc')
uas = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_uas_2005.nc')
vas = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_vas_2005.nc')
#psl = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p25/JRA55/regridded_raw/jra55do_v1p5_psl_2005.nc')

In [2]:
#cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src','python', 'afim_on_gadi.json'))
G_CICE    = xr.open_dataset('/g/data/jk72/da1339/grids/aom2_cice_0p25_deg.nc')#cice_prep.cice_grid_prep()
G_JRA55   = xr.open_dataset('/g/data/jk72/da1339/grids/JRA55do_grid_with_1yr_timeDim.nc')
F_weights = '/g/data/jk72/da1339/grids/weights/rmp_jrar_to_cict_BILINEAR.nc'
huss      = xr.open_dataset('/g/data/qv56/replicas/input4MIPs/CMIP6/OMIP/MRI/MRI-JRA55-do-1-5-0/atmos/3hrPt/huss/gr/v20200916/huss_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_200501010000-200512312100.nc')
rg        = xe.Regridder(G_JRA55, G_CICE, method='bilinear',  periodic=True, filename=F_weights, reuse_weights=True)
HUSS     = rg(huss)
HUSS.huss.isel(time=0).plot(figsize=(15,10))

ValueError: The horizontal shape of input data is (2920, 2), different from that of the regridder (320, 640)!

# Initial Conditions

## re-grid NCAR-gx1 initial conditions to 1/10-degree (or 1/4-degree) AOM2 grid
The first cell below should only need to be run once and then the next cell can be re-run to load and view the contents of the gridded data

In [ ]:
cice_prep        = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src','python', 'afim_on_gadi.json'))
G_CICE           = cice_prep.cice_grid_prep()
IC_GX1           = xr.open_dataset('sliced_gx1_v6.2005-01-01_added_grid.nc')
#F_weights        = '/g/data/jk72/da1339/grids/weights/map_NCAR-IC-1p0_AOM2-CICE_0p1_bilinear.nc'
F_weights        = '/g/data/jk72/da1339/grids/weights/map_NCAR-IC-1p0_AOM2-CICE_0p25_bilinear.nc'
rg               = xe.Regridder(IC_GX1, G_CICE, method='bilinear',  periodic=False, filename=F_weights, reuse_weights=False)
IC_NCAR         = rg(IC_GX1)
#IC_NCAR.to_netcdf('/scratch/jk72/da1339/cice-dirs/input/AFIM/ic/0p1/aom2_iceh_2005-01-01.nc')
IC_NCAR.to_netcdf('/scratch/jk72/da1339/cice-dirs/input/AFIM/ic/0p25/aom2_iceh_2005-01-01.nc')

In [ ]:
IC_NCAR = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/ic/0p25/aom2_iceh_2005-01-01.nc')

## add aditional fields that are required by CICE6 (as opposed to CICE5)
The additional fields can be seen if one 'looks' at the variables of 
This should be for the date of 01 Jan 2005 and make adjustments to coordinates and variable names to match the example initial conditions from NCAR

In [ ]:
#IC_AOM2 = xr.open_dataset('/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v140_iaf/output187/ice/OUTPUT/iceh.2004-12-daily.nc')
#IC_AOM2 = xr.open_dataset('/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ice/OUTPUT/iceh.2004-12.nc')
IC_AOM2 = xr.open_dataset('/g/data/jk72/da1339/afim_output/20230216_ACOM2_91days_dt_3600s_ndtd_20_scaled_qdp_scaled_hblt/restart/iced.2005-01-01-00000.nc')
#IC_AOM2 = IC_AOM2.isel(time=-1).drop('time').drop_vars('time_bounds')
#IC_AOM2 = IC_AOM2.swap_dims({'nj':'ny','ni':'nx','nc':'ncat'})
#IC_AOM2

### get sst and sss ocean fields 

In [ ]:
#salt = xr.open_dataset('/g/data/cj50/access-om2/raw-output/access-om2-01/01deg_jra55v140_iaf/output188/ocean/ocean-2d-surface_salt-1-daily-mean-ym_2005_01.nc')
ds = xr.open_dataset('/g/data/cj50/access-om2/raw-output/access-om2-025/025deg_jra55v13_ryf9091_gmredi6/output052/ocean/ocean.nc')
sss  = ds.isel(time=-1,st_ocean=0).salt.drop(['time','st_ocean'])
#temp = xr.open_dataset('/g/data/cj50/access-om2/raw-output/access-om2-01/01deg_jra55v140_iaf/output188/ocean/ocean-2d-surface_temp-1-daily-mean-ym_2005_01.nc')
sst  = ds.isel(time=-1,st_ocean=0).temp.drop(['time','st_ocean'])

## get atmosphere short-wave fields and do relevant computations

In [ ]:
msdwswrf              = cice_prep.era5_load_and_regrid(era5_var = 'msdwswrf', yr_str = '2005').resample(time='1D').mean('time')
mtdwswrf              = cice_prep.era5_load_and_regrid(era5_var = 'mtdwswrf', yr_str = '2005').resample(time='1D').mean('time')
SW_diff               = afim.compute_diffuse_sfc_em(msdwswrf.msdwswrf,mtdwswrf.mtdwswrf)

## load in CICE5 ice restart file from AOM2

In [ ]:
IC_ICE5 = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/runs/afim/restart/iced.2005-01-01-00000.nc')

In [ ]:
#hi: grid cell mean ice thickness (m)
IC_ICE5['hi']          = (("ny", "nx"), IC_AOM2.hi_m.data)
#hs: grid cell mean snow thickness (m)
IC_ICE5['hs']          = (("ny", "nx"), IC_AOM2.hs_m.data)
#congel: basal ice growth (m)
IC_ICE5['congel']      = (("ny", "nx"), IC_AOM2.congel_m.data)
#frazil: frazil ice growth (m)
IC_ICE5['frazil']      = (("ny", "nx"), IC_AOM2.frazil_m.data)
#snoice: snow–ice formation (m)
IC_ICE5['snoice']      = (("ny", "nx"), IC_AOM2.snoice_m.data)
#dvidtt: ice volume tendency due to thermodynamics (m/s)
IC_ICE5['dvidtt']      = (("ny", "nx"), IC_AOM2.dvidtt_m.data)
#dvidtd: ice volume tendency due to dynamics/transport (m/s)
IC_ICE5['dvidtd']      = (("ny", "nx"), IC_AOM2.dvidtd_m.data)
IC_ICE5['sst']          = (("ny", "nx"), sst.data-273)
IC_ICE5['sss']          = (("ny", "nx"), sss.data)
#frzmlt: freezing/melting potential (W/m^2)
IC_ICE5['frzmlt']       = (("ny", "nx"), IC_AOM2.frzmlt_m.data)

In [ ]:
IC_ICE5.attrs = {'istep1': 2773952,
 'time': 1483228800.0,
 'time_forc': 0.0,
 'nyr': 2005,
 'month': 1,
 'mday': 1,
 'sec': 0}
IC_ICE5

In [ ]:
IC_ICE5.to_netcdf('/scratch/jk72/da1339/cice-dirs/runs/afim_bran/restart/iced.2005-01-01-00000-MODIFIED.nc')

## CONSTRUCT IC NETCDF

In [ ]:
IC                = xr.Dataset()
IC.coords["TLON"] = (("ny", "nx"), IC_AOM2.TLON.data)
IC.coords["TLAT"] = (("ny", "nx"), IC_AOM2.TLAT.data)
IC.coords["ULON"] = (("ny", "nx"), IC_AOM2.ULON.data)
IC.coords["ULAT"] = (("ny", "nx"), IC_AOM2.ULAT.data)
IC.coords["NCAT"] = (("ncat"), IC_AOM2.ncat.data)
#tmask: land/boundary mask, thickness (T-cell)
IC['tmask']       = (("ny", "nx"), IC_AOM2.tmask.data)
#blkmask:
IC['blkmask']     = (("ny", "nx"), IC_AOM2.blkmask.data)
#tarea: area of T-cell (m^2)
IC['tarea']       = (("ny", "nx"), IC_AOM2.tarea.data)
#uarea: area of U-cell (m^2)
IC['uarea']       = (("ny", "nx"), IC_AOM2.uarea.data)
#dxt: width of T cell (∆𝑥) through the middle (m)
IC['dxt']         = (("ny", "nx"), IC_AOM2.dxt.data)
#dyt: width of T cell (∆y) through the middle (m)
IC['dyt']         = (("ny", "nx"), IC_AOM2.dyt.data)
#dxu: width of U cell (∆𝑥) through the middle (m)
IC['dxu']         = (("ny", "nx"), IC_AOM2.dxu.data)
#dyu: width of U cell (∆y) through the middle (m)
IC['dyu']         = (("ny", "nx"), IC_AOM2.dyu.data)
#HTN: length of northern edge (∆x) of T-cell
IC['HTN']         = (("ny", "nx"), IC_AOM2.HTN.data)
#HTE: length of eastern edge (∆𝑦) of T-cell
IC['HTE']         = (("ny", "nx"), IC_AOM2.HTE.data)
#ANGLE: for conversions between the POP grid and latitude- longitude grids (radians)
IC['ANGLE']       = (("ny", "nx"), IC_AOM2.ANGLE.data)
#ANGLET: ANGLE converted to T-cells
IC['ANGLET']      = (("ny", "nx"), IC_AOM2.ANGLET.data)
#hi: grid cell mean ice thickness (m)
IC['hi']          = (("ny", "nx"), IC_AOM2.hi.data)
#hs: grid cell mean snow thickness (m)
IC['hs']          = (("ny", "nx"), IC_AOM2.hs.data)
#congel: basal ice growth (m)
IC['congel']      = (("ny", "nx"), IC_AOM2.congel.data)
#frazil: frazil ice growth (m)
IC['frazil']      = (("ny", "nx"), IC_AOM2.frazil.data)
#snoice: snow–ice formation (m)
IC['snoice']      = (("ny", "nx"), IC_AOM2.snoice.data)
#dvidtt: ice volume tendency due to thermodynamics (m/s)
IC['dvidtt']      = (("ny", "nx"), IC_AOM2.dvidtt.data)
#dvidtd: ice volume tendency due to dynamics/transport (m/s)
IC['dvidtd']      = (("ny", "nx"), IC_AOM2.dvidtd.data)
#u(v)vel: x(y)-component of ice velocity
IC['uvel']        = (("ny", "nx"), IC_AOM2.uvel.data)
IC['vvel']        = (("ny", "nx"), IC_AOM2.vvel.data)
#scale_factor: scaling factor for shortwave radiation components
IC['scale_factor']= (("ny", "nx"), IC_ICE5.scale_factor.data)
#swv(i)dr(f): incoming shortwave radiation, visible (near IR), direct (diffuse)
IC['swvdr']       = (("ny", "nx"), IC_ICE5.swvdr.data)
IC['swvdf']       = (("ny", "nx"), IC_ICE5.swvdf.data)
IC['swidr']       = (("ny", "nx"), IC_ICE5.swidr.data)
IC['swidf']       = (("ny", "nx"), IC_ICE5.swidf.data)
#strocnx(y): ice–ocean stress in the x(y)-direction (U-cell)
IC['strocnxT']    = (("ny", "nx"), IC_ICE5.strocnxT.data)
IC['strocnyT']    = (("ny", "nx"), IC_ICE5.strocnyT.data)
#stressp(1-4): internal ice stress, 𝜎11 + 𝜎22
IC['stressp_1']   = (("ny", "nx"), IC_ICE5.stressp_1.data)
IC['stressp_2']   = (("ny", "nx"), IC_ICE5.stressp_2.data)
IC['stressp_3']   = (("ny", "nx"), IC_ICE5.stressp_3.data)
IC['stressp_4']   = (("ny", "nx"), IC_ICE5.stressp_4.data)
#stressm(1-4): internal ice stress, 𝜎11 − 𝜎22
IC['stressm_1']   = (("ny", "nx"), IC_ICE5.stressm_1.data)
IC['stressm_2']   = (("ny", "nx"), IC_ICE5.stressm_2.data)
IC['stressm_3']   = (("ny", "nx"), IC_ICE5.stressm_3.data)
IC['stressm_4']   = (("ny", "nx"), IC_ICE5.stressm_4.data)
#stress12(1-4): internal ice stress, 𝜎12
IC['stress12_1']   = (("ny", "nx"), IC_ICE5.stress12_1.data)
IC['stress12_2']   = (("ny", "nx"), IC_ICE5.stress12_2.data)
IC['stress12_3']   = (("ny", "nx"), IC_ICE5.stress12_3.data)
IC['stress12_4']   = (("ny", "nx"), IC_ICE5.stress12_4.data)
#iceumask: ice extent mask (U-cell)
IC['iceumask']     = (("ny", "nx"), IC_ICE5.iceumask.data)
#sst: sea-surface temperature (C)
IC['sst']          = (("ny", "nx"), sst.surface_temp.data-273)
IC['sss']          = (("ny", "nx"), sss.surface_salt.data)
#frzmlt: freezing/melting potential (W/m^2)
IC['frzmlt']       = (("ny", "nx"), IC_AOM2.frzmlt.data)
#frz_onset: day of year that freezing begins
#fsnow: snowfall rate (kg/m^2 /s)
#aice(n): total concentration of ice in grid cell (in category n)
IC['aice']         = (("ny", "nx"), IC_AOM2.aice.data)
IC['aicen']        = (("ncat","ny", "nx"), IC_AOM2.aicen.data)
#vice(n): volume per unit area of ice (in category n) (m)
IC['vicen']        = (("ncat","ny", "nx"), IC_AOM2.vicen.data)
#vso(n): volume per unit area of snow (in category n) (m)
IC['vsnon']        = (("ncat","ny", "nx"), IC_ICE5.vsnon.data)
#Tsfc(n): temperature of ice/snow top surface (in category n) (C)
IC['Tsfcn']        = (("ncat","ny", "nx"), IC_ICE5.Tsfcn.data)
#iage:
#IC['iage']         = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#FY:
#IC['FY']           = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#alvl:
#IC['alvl']         = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#apnd: area concentration of melt ponds
#IC['apnd']         = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#hpnd:
#IC['hpnd']         = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#dhs: depth difference for snow on sea ice and pond ice
#IC['dhs']          = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#ffrac: fraction of fsurfn used to melt pond ice
#IC['ffrac']        = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#fbrn:
#IC['fbrn']         = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#first_ice: flag for initial ice formation
#IC['first_ice']    = (("ncat","ny", "nx"), np.ones_like(IC_ICE5.Tsfcn.data)*np.nan)
#sice(001-7):
IC['sice001']      = (("ncat","ny", "nx"), IC_ICE5.sice001.data)
IC['sice002']      = (("ncat","ny", "nx"), IC_ICE5.sice002.data)
IC['sice003']      = (("ncat","ny", "nx"), IC_ICE5.sice003.data)
IC['sice004']      = (("ncat","ny", "nx"), IC_ICE5.sice004.data)
#qice(001-7)
IC['qice001']      = (("ncat","ny", "nx"), IC_ICE5.qice001.data)
IC['qice002']      = (("ncat","ny", "nx"), IC_ICE5.qice002.data)
IC['qice003']      = (("ncat","ny", "nx"), IC_ICE5.qice003.data)
IC['qice004']      = (("ncat","ny", "nx"), IC_ICE5.qice004.data)
#qsno001:
IC['qsno001']      = (("ncat","ny", "nx"), IC_ICE5.qsno001.data)

In [ ]:
IC.to_netcdf('/scratch/jk72/da1339/cice-dirs/input/AFIM/ic/0p1/IC_AOM2_CICE5_2005-01-01.nc')

In [ ]:
IC = xr.open_dataset('/scratch/jk72/da1339/cice-dirs/input/AFIM/ic/0p1/IC_AOM2_CICE5_2005-01-01.nc')
OC = xr.open_dataset('/scratch/jk72/da1339/model_input/ACOM2/acom2_ocn_frcg_cice6_0p1_2005.nc')

# AOM2 coallation for stand-alone forcing of CICE6

In [ ]:
importlib.reload(afim)

In [ ]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
cice_prep.acom2_coallate_for_cice_forcing()

# construct yearly OCN dataset from AC-OM2-01
1. request data using cosima cookbook
2. compute sea surface slope using eta 
3. pull in the qdp ... requires compute_ocean_qdp_from 

In [ ]:
geolon  = xr.open_dataset('/g/data/ik11/grids/ocean_grid_01.nc').geolon_t.swap_dims({'grid_y_T':'nj','grid_x_T':'ni'})
geolat  = xr.open_dataset('/g/data/ik11/grids/ocean_grid_01.nc').geolat_t.swap_dims({'grid_y_T':'nj','grid_x_T':'ni'})

In [ ]:
session = cc.database.create_session()
eta     = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='sea_level',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
sst     = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='surface_temp',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
sss     = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='surface_salt',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
mld     = xr.full_like(sst,50)#cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='mld',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
u       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='u',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00').isel(st_ocean=0)
v       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='v',session=session, frequency='1 monthly', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00').isel(st_ocean=0)
qdp     = xr.full_like(sst,0)
G_CICE  = xr.open_dataset(cice_prep.F_G_CICE_original)
dhdx    = afim.compute_ocn_sfc_slope(eta, G_CICE.hte.data, direction='x')
dhdy    = afim.compute_ocn_sfc_slope(eta, G_CICE.htn.data, direction='y')

In [ ]:
lat_mom = sst.yt_ocean
lon_mom = sst.xt_ocean
G_tmp   = xr.Dataset({'lon':lon_mom,'lat':lat_mom})
F_net   = cice_prep.compute_era5_net_atmospheric_surface_heat_flux(G_dst              = G_tmp,
                                                              yr_str             = str(2005),
                                                              mean_type          = 'monthly',
                                                              regrid_method      = cice_prep.ERA5_regrid_method,
                                                              regrid_periodicity = cice_prep.ERA5_periodicity,
                                                              cnk_dict           = cice_prep.ERA5_chunking,
                                                              generate_weights   = cice_prep.ERA5_regen_weights,
                                                              weights_file_name  = cice_prep.ERA5_regen_weights)
mld_k  = mld.astype(np.single).load()
Fnet_k = F_net.astype(np.single).load()
print("\t\tslicing temporal temperature derivative at the mixed layer depth")
dTdt_k = dTdt.sel(st_ocean=mld_k,method='nearest').drop('st_ocean').astype(np.single).load()
print("\t\tslicing temporal temperature at the mixed layer depth")
temp_k = temp.sel(st_ocean=mld_k,method='nearest').drop('st_ocean').astype(np.single).load()
print("\t\tslicing salinity temperature at the mixed layer depth")
salt_k = salt.sel(st_ocean=mld_k,method='nearest').drop('st_ocean').astype(np.single).load()
print("\t\tcomputing ocean heat capacity at the mixed layer depth")
cp_k = afim.compute_ocn_heat_capacity_at_depth(salt_k, temp_k, mld_k, mld.yt_ocean).astype(np.single)
cp_k = cp_k.load()
print("\t\tcomputing ocean density at the mixed layer depth")
rho_k = afim.compute_ocn_density_at_depth(salt_k, temp_k, mld_k, mld.yt_ocean).astype(np.single)
rho_k = rho_k.load()
print("\t\tcomputing ocean heat flux at the mixed layer depth")
qdp_k = compute_ocn_heat_flux_at_depth(rho_k, cp_k, mld_k, Fnet_k, dTdt_k)
qdp_k = qdp_k.load()
qdp_k = qdp_k.assign_coords(time=temp.time.isel(time=k).values).expand_dims('time').astype(np.single).to_dataset(name='qdp')

In [ ]:
interp_dates = pd.date_range(start='2005-01-01 12:00',periods=365,freq='1D')
u_int    = u.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
v_int    = v.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
mld_int  = mld.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})*0
sss_int  = sss.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
sst_int  = sst.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
dhdx_int = dhdx.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
dhdy_int = dhdy.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
qdp_int  = qdp.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})
mld_int  = mld.interp(time=interp_dates,method="quadratic",kwargs={"fill_value": "extrapolate"})

In [ ]:
data_vars = {'qdp' : (['time','nj','ni'],qdp_int.data,{'units':'W/m^2','long_name':'deep ocean heat flux','_FillValue':-2e8}),
             'T'   : (['time','nj','ni'],sst_int.data,{'units':'K','long_name':'sea surface temperature','_FillValue':-2e8}),
             'S'   : (['time','nj','ni'],sss_int.data,{'units':'psu','long_name':'sea surface salinity','_FillValue':-2e8}),
             'hblt': (['time','nj','ni'],mld_int.data,{'units':'m','long_name':'mixed layer depth','_FillValue':-2e8}),
             'U'   : (['time','nj','ni'],u_int.data,{'units':'m/s','long_name':'meridional surface current','_FillValue':-2e8}),
             'V'   : (['time','nj','ni'],v_int.data,{'units':'m/s','long_name':'zonal surface current','_FillValue':-2e8}),
             'dhdx': (['time','nj','ni'],dhdx_int.data,{'units':'m','long_name':'zonal sea surface slope','_FillValue':-2e8}),
             'dhdy': (['time','nj','ni'],dhdy_int.data,{'units':'m','long_name':'meridional sea surface slope','_FillValue':-2e8})}
coords = {'xc'   : (['nj','ni'],geolon.data,{'units':'degrees_east'}),
          'yc'   : (['nj','ni'],geolat.data,{'units':'degrees_north'}),
          'time' : (['time'],sst_int.time.data,{'long_name':'time','cartesian_axis':'T','calendar_type':'GREGORIAN'})}
attrs = {'creation_date': datetime.now().strftime('%Y-%m-%d %H'),
         'conventions'  : "CCSM data model domain description -- for CICE6 standalone 'NCAR' ocean option",
         'title'        : "daily averaged ocean forcing from ACCESS-OM2-01 output for CICE6 standalone ocean forcing",
         'source'       : "ACCESS-OM2-01",
         'comment'      : "source files found on gadi, /g/data/cj50/access-om2/raw-output/access-om2-01/01deg_jra55v140_iaf",
         'calendar'     : "standard",
         'note1'        : "grid is ACCESS-OM2 t-grid",
         'note2'        : "u,v are left un-interpolated to t-grid (i.e. they are from ACCESS-OM2 u-grid)",
         'note3'        : "source files were concatenated using NCO tools, see python afim module method concat_access_om2",
         'note4'        : "dhdx,dhdy are described in ACCESS-OM2 output as effective sea level (eta_t + patm/(rho0*g)) on T cells",
         'note4a'       : "I have computed sea surface slope using python afim module function compute_ocn_sfc_slope, which assumes input is eta_t",
         'author'       : 'Daniel P Atwater',
         'email'        : 'daniel.atwater@utas.edu.au'}
F_OCN     = '/scratch/jk72/da1339/cice-dirs/input/AFIM/forcing/0p1/monthly/acom2_ocn_frcg_cice6_0p1_2005_zeroed_qdp_scalar50m_hblt_scipy_interp.nc'
OCN       = xr.Dataset(data_vars=data_vars,coords=coords,attrs=attrs)
enc_dict  = {'shuffle':True,'zlib':True,'complevel':5}
write_job = OCN.to_netcdf(F_OCN, unlimited_dims=['time'], compute=False,
                          encoding={'qdp':enc_dict, 'T':enc_dict, 'S':enc_dict, 'hblt':enc_dict,
                                    'U':enc_dict, 'V':enc_dict, 'dhdx':enc_dict, 'dhdy':enc_dict})
#write_job = OCN.to_netcdf(F_OCN, unlimited_dims=['time'], compute=True,
#                          encoding={'qdp':enc_dict, 'T':enc_dict, 'S':enc_dict, 'hblt':enc_dict,
#                                    'U':enc_dict, 'V':enc_dict, 'dhdx':enc_dict, 'dhdy':enc_dict})
with ProgressBar():
    print(f"Writing to {F_OCN}")
    write_job.compute()

In [ ]:
sst.isel(time=180).plot(figsize=(20,10))
sss.isel(time=180).plot(figsize=(20,10))
mld.isel(time=180).plot(figsize=(20,10))
u.isel(time=180).plot(figsize=(20,10))
v.isel(time=180).plot(figsize=(20,10))
dhdx.isel(time=180).plot(figsize=(20,10))
dhdy.isel(time=180).plot(figsize=(20,10))

In [ ]:
xr.open_dataset('/scratch/jk72/da1339/qdp/tmp/qdp_yd129.nc').qdp.plot(figsize=(20,10))

In [ ]:
variables = cc.querying.get_variables(session, experiment='01deg_jra55v140_iaf')
list(variables.name)

In [ ]:
IC_gx3 = xr.open_dataset('/g/data/jk72/da1339/cice-dirs/input/CICE_data/ic/gx3/iced_gx3_v6.2005-01-01.nc')
G_gx3 = xr.open_dataset('/g/data/jk72/da1339/cice-dirs/input/CICE_data/grid/gx3/grid_gx3.nc')

In [ ]:
IC_gx3['lat'] = (['nj','ni'],G_gx3.ulat.values*(180/np.pi),{'units':'degrees_north','_FillValue':-2e8})
IC_gx3['lon'] = (['nj','ni'],G_gx3.ulon.values*(180/np.pi),{'units':'degrees_east','_FillValue':-2e8})
IC_gx3['htn'] = (['nj','ni'],G_gx3.htn.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['hte'] = (['nj','ni'],G_gx3.hte.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['hus'] = (['nj','ni'],G_gx3.hus.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['angle'] = (['nj','ni'],G_gx3.angle.values*(180/np.pi),{'units':'degrees','_FillValue':-2e16})

In [ ]:
G_CICE        = xr.open_dataset(cice_prep.F_G_CICE_original)
G_CICE['lat'] = (['ny','nx'],G_CICE.tlat.values*(180/np.pi),{'units':'degrees_north','_FillValue':-2e8})
G_CICE['lon'] = (['ny','nx'],G_CICE.tlon.values*(180/np.pi),{'units':'degrees_east','_FillValue':-2e8})
G_CICE        = G_CICE.drop(('ulat','ulon','tlat','tlon','clon_t','clat_t','clat_u','clon_u','angle','uarea'))
G_CICE        = G_CICE.assign_coords(xt_ocean=G_CICE['lon'][0,:],yt_ocean=G_CICE['lat'][:,0])
G_CICE        = G_CICE.set_index(nx='xt_ocean',ny='yt_ocean')

In [ ]:
rg = xe.Regridder(IC_gx3, G_CICE, method='patch')
IC = rg(IC_gx3)

In [ ]:
IC.aicen.isel(ncat=0).plot(figsize=(20,10))

In [ ]:
G_BRAN         = xr.open_dataset("/g/data/gb6/BRAN/BRAN2020/static/ocean_grid.nc")
LN,LT          = np.meshgrid(G_BRAN.xt_ocean.values,G_BRAN.yt_ocean.values)
G_BRANt['lon'] = (['yt_ocean','xt_ocean'],LN,{'units':'degrees_east'})
G_BRANt['lat'] = (['yt_ocean','xt_ocean'],LT,{'units':'degrees_north'})
G_BRANt        = G_BRANt.drop(('xu_ocean','yu_ocean','hu','kmu','umask','tmask','st_edges_ocean','Time','st_ocean'))
G_BRANt

In [ ]:
rg = xe.Regridder(G_BRAN, G_CICE, 'bilinear', filename='/g/data/jk72/da1339/grids/weights/bran_ac_om2_01_bilinear.nc', reuse_weights=False)

In [ ]:
start_date = cice_prep.regrid_start_date
dt_obj = datetime.strptime(start_date,'%Y-%m-%d').date() + pd.DateOffset(years=0)
spdts  = cice_prep.time_series_day_month_start_or_end(start_date=dt_obj.strftime('%Y-%m-%d'),month_start=False)
stdts  = cice_prep.time_series_day_month_start_or_end(start_date=dt_obj.strftime('%Y-%m-%d'))
yr_str = cice_prep.define_year_string(stdts[0])
G_CICE = cice_prep.cice_grid_prep()
G_BRAN = cice_prep.bran_grid_prep()
rg     = xe.Regridder(G_BRAN, G_CICE, method='bilinear',  periodic=True,  filename=cice_prep.F_BRAN_weights, reuse_weights=True)
temp_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_temp_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
temp   = rg(temp_n)
temp   = temp.drop(('nv'))
salt_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_salt_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
salt   = rg(salt_n)
salt   = salt.drop(('nv'))
mld_n  = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_mld_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
mld    = rg(mld_n)
mld    = mld.drop('nv')
uocn_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_u_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
uocn_n = uocn_n.isel(st_ocean=0)
uocn_n = uocn_n.rename(xu_ocean='xt_ocean',yu_ocean='yt_ocean')
uocn   = rg(uocn_n)  
uocn   = uocn.drop(('st_ocean','nv','st_edges_ocean'))
vocn_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_v_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
vocn_n = vocn_n.isel(st_ocean=0)
vocn_n = vocn_n.rename(xu_ocean='xt_ocean',yu_ocean='yt_ocean')
vocn   = rg(vocn_n)
vocn   = vocn.drop(('st_ocean','nv','st_edges_ocean'))
eta_n  = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_eta_t_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
eta    = rg(eta_n)
eta    = eta.drop(('nv'))
sst    = temp.sel(st_ocean=0,method='nearest')
sss    = salt.sel(st_ocean=0,method='nearest')
dhdx   = afim.compute_ocn_sfc_slope(eta, G_CICE, direction='x', grid_scale_factor=100)
dhdy   = afim.compute_ocn_sfc_slope(eta, G_CICE, direction='y', grid_scale_factor=100)
# compute qdp
print("computing temperature derivative over time")
dTdt    = temp.differentiate("Time")
print("computing atmospheric surface heat flux")
G_ERA5    = cice_prep.era5_grid_prep()
rg        = xe.Regridder(G_ERA5, G_CICE, method='bilinear', periodic=True, filename=cice_prep.F_ERA5_weights, reuse_weights=True)
dt0       = datetime.strptime(start_date,'%Y-%m-%d') + timedelta(days=(0*365))
lw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwlwrf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
sw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwswrf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
msshf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msshf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
mslhf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'mslhf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
F_net_nat = sw_nat.msdwswrf - lw_nat.msdwlwrf - mslhf_nat.mslhf - msshf_nat.msshf
F_net_rg  = rg(F_net_nat)
F_net_rg  = F_net_rg.assign_coords(lon=G_CICE.lon[0,:],lat=G_CICE.lat[:,0])
F_net_rg  = F_net_rg.rename({'ny':'yt_ocean','nx':'xt_ocean'})
F_net_rg  = F_net_rg.resample(time='1D').mean()
print("computing ocean heat capacity at the mixed layer depth")
cp   = afim.compute_ocn_heat_capacity_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)
#cp   = cp.drop(('st_ocean','nv','st_edges_ocean'))
print("computing ocean density at the mixed layer depth")
rho  = afim.compute_ocn_density_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)
#rho  = rho.drop(('st_ocean','nv','st_edges_ocean'))

In [ ]:
rho  = afim.compute_ocn_density_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)

In [ ]:
cp.isel(st_ocean=0,Time=0).plot(figsize=(20,10))

In [ ]:
rho.isel(st_ocean=0,Time=0).plot(figsize=(20,10))

In [ ]:
start_date         = cice_prep.BRAN_start_date
D_BRAN             = cice_prep.D_BRAN
regrid_method      = cice_prep.BRAN_regrid_method
regrid_periodicity = cice_prep.BRAN_periodicity
var_freq           = cice_prep.BRAN_var_freq
F_weights          = cice_prep.F_BRAN_t_weights
cnk_dict           = cice_prep.BRAN_t_chunking
G_BRAN             = cice_prep.bran_grid_prep()
G_dst              = cice_prep.cice_grid_prep()
dt_obj             = cice_prep.define_datetime_object(start_date=start_date, year_offset=0)
yr_str             = cice_prep.year_string_from_datetime_object(dt_obj)
mo_str             = '{:02d}'.format(0+1)

In [ ]:
fig = plt.figure(figsize=(15,12))
axs = plt.scatter(G_BRAN.lon,G_BRAN.lat)

In [ ]:
fig = plt.figure(figsize=(15,12))
axs = plt.scatter(G_dst.lon,G_dst.lat)

In [ ]:
rg = xe.Regridder(G_BRAN, G_dst, method=regrid_method,  periodic=regrid_periodicity, filename=F_weights, reuse_weights=False)

# construct yearly OCN dataset from BRAN

1. request data using cosima cookbook
2. compute sea surface slope using eta
3. pull in the qdp ... requires compute_ocean_qdp_from

In [ ]:
cice_prep.bran_load_and_regrid( bran_var='temp' , yr_str='2005', mo_str='05', grid_type='t')

In [ ]:
session = cc.database.create_session()
pd.set_option("display.max_rows", None)
cc.querying.get_experiments(session)